# 05 · Aggregate results and build the report
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/05_report.ipynb)

Collects every saved result into `report.md` (tables + figures), `analysis/summary.json`, and a
zip of the lightweight artifacts (configs, metrics, predictions, logs, figures; adapters excluded)
that can be committed to the repository's `results/` folder.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
GPU_PROFILE = "auto"  #@param ["auto", "t4", "a100"]
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if any(k in GPU for k in ("A100", "H100")) else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"  # different budgets/sizes: keep results of the two profiles apart
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} | profile: {GPU_PROFILE} | outputs -> {OUT}")
print("configs:", CONFIG_FILES)

In [ ]:
!python -m cegqa.cli report $CONFIG

In [ ]:
from IPython.display import Markdown, display
import os
os.chdir(OUT)
display(Markdown(open("report.md").read()))
os.chdir(REPO_DIR)

In [ ]:
!python scripts/export_results.py --src $OUT --dst /content/results
!cd /content && zip -qr cegqa_results.zip results && ls -lh /content/cegqa_results.zip
from google.colab import files
files.download("/content/cegqa_results.zip")